In [2]:
import uuid
import numpy as np
from sklearn.cluster import KMeans
from typing import List, Dict

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_62497/1866507504.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
# 1. Setup Local Embeddings
embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio", 
    model="text-embedding-nomic-embed-text-v1.5",
    check_embedding_ctx_length=False 
)

# 2. Setup Local LLM for Summarization
# Make sure the model name matches whatever text-generation model you have loaded in LM Studio
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.1, 
    model="your-local-llm-name" 
)

In [3]:
def get_embeddings(texts: List[str]) -> np.ndarray:
    """Fetches embeddings for a list of texts."""
    emb_list = embeddings.embed_documents(texts)
    return np.array(emb_list)

def perform_clustering(texts: List[str], num_clusters: int = 3) -> Dict[int, List[str]]:
    """Clusters texts based on their semantic embeddings."""
    if len(texts) <= num_clusters:
        return {0: texts} # Not enough texts to cluster, group them all together
        
    vectors = get_embeddings(texts)
    
    # Run K-Means clustering
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    labels = kmeans.fit_predict(vectors)
    
    # Group the actual texts by their cluster label
    clusters = {}
    for text, label in zip(texts, labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(text)
        
    return clusters

In [4]:
# Prompt to generate cluster summaries
summary_prompt = PromptTemplate.from_template(
    """You are an expert summarizer. Below is a set of text documents from a larger corpus. 
    Write a comprehensive summary of these documents, highlighting the main themes, key entities, and relationships.
    
    DOCUMENTS:
    {context}
    
    SUMMARY:"""
)

# The summarization pipeline
summary_chain = summary_prompt | llm | StrOutputParser()

def summarize_cluster(cluster_texts: List[str]) -> str:
    """Feeds a cluster of texts to the LLM to generate a summary."""
    # Join the cluster texts into a single string
    context = "\n\n---\n\n".join(cluster_texts)
    
    # Invoke the LLM
    return summary_chain.invoke({"context": context})

In [13]:
def build_raptor_tree(docs: List[Document], max_levels: int = 3, num_clusters: int = 3) -> List[Document]:
    """
    Recursively builds the RAPTOR tree.
    Returns a flattened list of ALL documents (original leaves + summaries at all levels).
    """
    all_documents = []
    current_level_texts = [doc.page_content for doc in docs]
    
    # Add the base leaf nodes to our final list
    all_documents.extend(docs)
    
    for level in range(1, max_levels + 1):
        print(f"Processing Level {level}...")
        
        # Stop if we've summarized down to a single chunk
        if len(current_level_texts) <= 1:
            print("Tree consolidated to a single root node. Stopping.")
            break
            
        # 1. Cluster the current texts
        clusters = perform_clustering(current_level_texts, num_clusters=min(num_clusters, len(current_level_texts)))
        
        # 2. Summarize each cluster to create the next level
        next_level_texts = []
        for label, texts in clusters.items():
            print(f"  Summarizing cluster {label} ({len(texts)} chunks)...")
            summary = summarize_cluster(texts)
            next_level_texts.append(summary)
            
            # 3. Store the summary as a Document instance
            # FIX: Cast the numpy.int32 label to a standard Python int
            summary_doc = Document(
                page_content=summary, 
                metadata={
                    "raptor_level": level, 
                    "cluster_id": int(label) 
                }
            )
            all_documents.append(summary_doc)
            
        # Move up the tree
        current_level_texts = next_level_texts
        
    return all_documents

In [14]:
lakehouse_architecture = """
The Data Lakehouse architecture represents a paradigm shift in data engineering, combining the flexibility and scalability of data lakes with the ACID transactions and performance of data warehouses. By implementing open table formats like Delta Lake, Apache Iceberg, or Apache Hudi, organizations can bring reliability directly to data stored in cheap object storage like AWS S3.

In a modern Lakehouse setup, data ingestion is often handled through a medallion architecture. Raw data lands in the Bronze layer, often ingested via automated pipelines or streaming tools like Apache Spark. This layer acts as an immutable historical archive. The Silver layer cleans, filters, and conforms this data into a queryable state. Finally, the Gold layer aggregates the data for business-level reporting, powering BI dashboards and downstream machine learning models.

Orchestration is a critical component of maintaining this pipeline. Tools like dbt (data build tool) allow engineers to transform data in the warehouse using simple SQL SELECT statements, while managing dependencies and testing natively. When combined with compute engines like Databricks, engineers can seamlessly transition between Python, PySpark, and SQL depending on the specific workload requirements.

However, moving to a Lakehouse is not without challenges. Managing compute costs on AWS or Azure requires strict tagging, auto-scaling policies, and right-sizing of clusters. Additionally, while the separation of storage and compute allows for infinite scaling, poorly partitioned data can lead to massive "file scan" bottlenecks during querying, requiring regular vacuuming and optimization routines.
"""

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Choose which text to test
raw_text = lakehouse_architecture 

# Create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Small chunks to force the clustering algorithm to work
    chunk_overlap=20,
    separators=["\n\n", "\n", ".", " "]
)

# Convert to LangChain Document objects
leaf_docs = text_splitter.create_documents([raw_text])
print(f"Created {len(leaf_docs)} initial leaf chunks.")

# Now you can pass 'leaf_docs' directly into the build_raptor_tree() function!

Created 13 initial leaf chunks.


In [16]:
# 1. Load and chunk your initial text (replace with your actual data loader)
# raw_text = "Your very long document text goes here..."

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000, 
#     chunk_overlap=100
# )
# leaf_docs = text_splitter.create_documents([raw_text])
# print(f"Created {len(leaf_docs)} initial leaf chunks.")

# 2. Run the RAPTOR engine
# Note: This will make several calls to your local LM Studio instance!
flattened_raptor_docs = build_raptor_tree(leaf_docs, max_levels=3, num_clusters=4)
print(f"Total documents to index (leaves + summaries): {len(flattened_raptor_docs)}")

# 3. Index everything into Chroma
vectorstore = Chroma.from_documents(
    documents=flattened_raptor_docs,
    embedding=embeddings,
    collection_name="raptor_tree"
)

# 4. Setup the standard retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Test it out!
query = "What is the overarching theme of the document?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    level = doc.metadata.get('raptor_level', 0) # 0 means it's an original leaf chunk
    print(f"\nResult {i+1} (Level {level}):\n{doc.page_content[:200]}...")

Processing Level 1...
  Summarizing cluster 2 (4 chunks)...
  Summarizing cluster 1 (3 chunks)...
  Summarizing cluster 0 (4 chunks)...
  Summarizing cluster 3 (2 chunks)...
Processing Level 2...
  Summarizing cluster 0 (4 chunks)...
Processing Level 3...
Tree consolidated to a single root node. Stopping.
Total documents to index (leaves + summaries): 18

Result 1 (Level 1):
Based on the fragmented documents, the core theme is the **critical necessity of robust maintenance protocols for complex data or operational pipelines.**

### Comprehensive Summary

The primary focus...

Result 2 (Level 0):
Your very long document text goes here......

Result 3 (Level 0):
Your very long document text goes here......

Result 4 (Level 0):
Your very long document text goes here......

Result 5 (Level 0):
....


In [3]:
from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

ModuleNotFoundError: No module named 'ragatouille'

In [ ]:
import requests

def get_wikipedia_page(title: str):
    """
    Retrieve the full text content of a Wikipedia page.

    :param title: str - Title of the Wikipedia page.
    :return: str - Full text content of the page as raw string.
    """
    # Wikipedia API endpoint
    URL = "https://en.wikipedia.org/w/api.php"

    # Parameters for the API request
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    # Custom User-Agent header to comply with Wikipedia's best practices
    headers = {"User-Agent": "RAGatouille_tutorial/0.0.1 (ben@clavie.eu)"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()

    # Extracting page content
    page = next(iter(data["query"]["pages"].values()))
    return page["extract"] if "extract" in page else None

full_document = get_wikipedia_page("Hayao_Miyazaki")

In [ ]:
RAG.index(
    collection=[full_document],
    index_name="Miyazaki-123",
    max_document_length=180,
    split_documents=True,
)